In [ ]:
import re
from collections import Counter
from pathlib import Path

import yaml
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

KB_DIR = Path("../data/knowledge_base/generated")   # notebook runs from RAG_pipeline/
CITIES = ["Delhi", "Pune", "Kolkata"]               # first iteration: 3 cities only
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CHUNK_TOKENS = 450                                  # bge-small reads max 512 tokens
OVERLAP_TOKENS = 50


: 

In [11]:
def load_md(path):
    raw = path.read_text(encoding="utf-8")
    _, front, body = raw.split("---", 2)            # file starts with ---\n<yaml>\n---
    meta = yaml.safe_load(front)

    # Chroma accepts only str / int / float / bool (YAML turns 2024-05-01 into a date)
    meta = {k: v if isinstance(v, (str, int, float, bool)) else str(v) for k, v in meta.items()}
    meta.pop("source", None)                        # same long string in every file

    body = body.strip()
    meta["path"] = str(path.relative_to(KB_DIR))
    meta["title"] = re.search(r"^# (.+)$", body, re.M).group(1)
    return {"text": body, "metadata": meta}


In [12]:
docs = []
for path in sorted(KB_DIR.rglob("*.md")):
    doc = load_md(path)
    city = doc["metadata"].get("city")
    if city in CITIES or doc["metadata"]["doc_type"] in ("dataset_card", "reference"):
        docs.append(doc)

print("Documents loaded:", len(docs))
print(Counter(d["metadata"].get("city", "(no city)") for d in docs))
print(Counter(d["metadata"]["doc_type"] for d in docs))


Documents loaded: 528
Counter({'Delhi': 175, 'Kolkata': 175, 'Pune': 175, '(no city)': 3})
Counter({'monthly_summary': 360, 'seasonal_summary': 117, 'event_log': 33, 'city_profile': 3, 'event_summary': 3, 'rainy_season': 3, 'records': 3, 'trends': 3, 'reference': 2, 'dataset_card': 1})


In [13]:
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=CHUNK_TOKENS,
    chunk_overlap=OVERLAP_TOKENS,
    separators=["\n## ", "\n### ", "\n\n", "\n"],
)

chunks = []
for doc in docs:
    title = doc["metadata"]["title"]
    for i, text in enumerate(splitter.split_text(doc["text"])):
        if not text.startswith("# "):               # later pieces: say which city/year they belong to
            text = f"# {title}\n\n{text}"
        chunks.append({
            "id": f"{doc['metadata']['path']}#{i}",
            "text": text,
            "metadata": {**doc["metadata"], "chunk": i},
        })

print("Chunks:", len(chunks))


Chunks: 546


In [24]:
print(chunks[0]['text'])


# City climate profile: Delhi

Delhi (NCT of Delhi) is an inland city on the Indo-Gangetic plains, at latitude 28.58°N, longitude 77.19°E
and an elevation of about 217 m. For heatwave and cold-wave rules it is treated as a **plains** location.

Based on 2015–2024 data, the hottest month in Delhi is **May**
(average maximum 38.9°C) and the coldest month is **January**
(average minimum 7.8°C). Normal annual rainfall is about **762 mm**.
Most of its rain falls in the southwest monsoon (June–September); the wettest month is July (249 mm on average) and the driest
is November (6 mm). Months that each bring at least 10% of the annual rain:
July, August, September.


In [25]:
sizes = [len(tokenizer.encode(c["text"])) for c in chunks]
per_doc = Counter(c["metadata"]["path"] for c in chunks)

print("Tokens per chunk: min", min(sizes), "| max", max(sizes))
print("Chunks over 512 tokens:", sum(s > 512 for s in sizes))
print("Docs kept as 1 chunk:", sum(n == 1 for n in per_doc.values()), "| docs split:", sum(n > 1 for n in per_doc.values()))
print("Unique ids:", len({c["id"] for c in chunks}) == len(chunks))
print("Metadata types ok:", all(isinstance(v, (str, int, float, bool)) for c in chunks for v in c["metadata"].values()))


Tokens per chunk: min 49 | max 460
Chunks over 512 tokens: 0
Docs kept as 1 chunk: 513 | docs split: 15
Unique ids: True
Metadata types ok: True


In [34]:
from langchain_core.documents import Document

documents = [Document(page_content=c["text"], metadata=c["metadata"], id=c["id"]) for c in chunks]
print("LangChain documents:", len(documents))


LangChain documents: 546


In [37]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embedding_model = HuggingFaceBgeEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
    query_instruction="Represent this sentence for searching relevant passages: ",
)


NameError: name 'nn' is not defined

In [31]:
from langchain_chroma import Chroma
import shutil, os, time

VECTOR_DIR = "../vectorstore/chroma"
COLLECTION = "weather_kb"

if os.path.isdir(VECTOR_DIR):
    shutil.rmtree(VECTOR_DIR)     # rebuild fresh every run

t = time.time()
vectorstore = Chroma.from_documents(
    documents=document,
    embedding=embedding_model,
    collection_name=COLLECTION,
    persist_directory=VECTOR_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)
print("Stored:", vectorstore._collection.count(), f"{time.time()-t:.0f}s")


NameError: name 'embedding_model' is not defined